# Phase 06C — EnViT5 pretrained baseline EN→VI

Notebook này ghi nhận baseline EN→VI của `VietAI/envit5-translation` trước khi áp dụng miền CNTT. General Test và IT Test đã sealed ở Phase 05; protocol v2, metric và workload giữ nguyên để so sánh công bằng với OPUS-MT.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs').is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / 'src'))

from core_mt.adapters.envit5 import EnViT5Adapter
from core_mt.contracts import Direction
from core_mt.data import load_locked_evaluation_set
from core_mt.protocol import load_frozen_protocol

CONFIG_PATH = ROOT / 'configs' / 'envit5_baseline_v1.json'
PROTOCOL_PATH = ROOT / 'configs' / 'evaluation_protocol_v2.json'
RUN_ROOT = ROOT / 'runs' / 'core_mt_baseline_v2' / 'envit5'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
protocol = load_frozen_protocol(PROTOCOL_PATH)
print(f'Protocol: {protocol.protocol_version}; dataset release: {protocol.dataset_release}')

Protocol: core_mt_baseline_v2; dataset release: phase05_release_v1


## 1. Phần riêng của EnViT5 EN→VI

EnViT5 dùng một checkpoint T5 cho cả hai chiều. Adapter truyền câu nguồn dưới dạng `en: <source>` theo model card, sau đó chỉ gỡ target tag `vi:` nếu decoder sinh lại tag đó. Shared CORE vẫn quyết định test set, seed, beam size, metric, latency sample và artifact.

In [2]:
import core_mt.adapters.envit5 as envit5_module
display(Markdown(envit5_module.__doc__.strip()))
display(Markdown(EnViT5Adapter.__doc__.strip()))

Adapter EnViT5: phần riêng của checkpoint T5 song ngữ VietAI.

EnViT5 là một checkpoint dùng cho cả EN→VI và VI→EN. Chiều dịch được chỉ ra
bằng tiền tố input `en:` hoặc `vi:` theo model card, khác với OPUS-MT dùng hai
checkpoint Marian. Decoder có thể trả lại tiền tố target; adapter chỉ gỡ đúng
token giao diện này trước khi runner chấm, không làm sạch nội dung bản dịch.

Bọc `VietAI/envit5-translation` theo interface TranslationAdapter chung.

In [3]:
pd.DataFrame([
    {
        'direction': direction,
        'checkpoint': spec['model_id'],
        'requested_revision': spec['revision'],
        'input_prefix': spec['source_prefix'],
        'decoded_target_tag_removed': spec['target_prefix'],
    }
    for direction, spec in config['directions'].items()
])

,direction,checkpoint,requested_revision,input_prefix,decoded_target_tag_removed
0,en_to_vi,VietAI/envit5-translation,main,en:,vi:
1,vi_to_en,VietAI/envit5-translation,main,vi:,en:


## 2. Kiểm tra input EN→VI trước tokenizer

Đây là kiểm tra format, chưa tải checkpoint. Model card dùng `en:` để cho EnViT5 biết ngôn ngữ nguồn. `normalize_output()` chỉ gỡ `vi:` ở đầu output khi model sinh lại đúng tag giao diện đó; nội dung còn lại giữ nguyên để chấm metric.

In [4]:
source = 'Install the package from the repository.'
adapter_preview = EnViT5Adapter(Direction.EN_TO_VI, CONFIG_PATH, 'cpu')
prepared = adapter_preview.prepare_input(source)
decoded_demo = 'vi: translation example'
pd.DataFrame([{
    'direction': Direction.EN_TO_VI.value,
    'checkpoint': config['directions']['en_to_vi']['model_id'],
    'source': source,
    'input_to_tokenizer': prepared,
    'decoded_demo': decoded_demo,
    'text_scored': adapter_preview.normalize_output(decoded_demo),
}])

,direction,checkpoint,source,input_to_tokenizer,decoded_demo,text_scored
0,en_to_vi,VietAI/envit5-translation,Install the package from the repository.,en: Install the package from the repository.,vi: translation example,translation example


## 3. Protocol benchmark v2

Chất lượng và hiệu năng là hai phép đo khác nhau. Mỗi câu của General Test và IT Test được sinh bản dịch đúng một lần để tính chrF++ và SacreBLEU. Sau đó, mỗi test set chọn tối đa 128 câu bằng SHA-256 của `seed:row_id`, warm-up tối đa 32 câu, rồi đo 5 lượt trên mẫu cố định. Vì vậy benchmark giữ được tính lặp lại nhưng không dịch lặp toàn bộ corpus năm lần.

In [5]:
datasets = {name: load_locked_evaluation_set(name, ROOT) for name in protocol.evaluation_sets}
workload = pd.DataFrame([
    {
        'test_set': name,
        'quality_sentences': len(dataset.rows),
        'quality_passes': protocol.benchmark.quality_passes,
        'latency_sample_max': min(len(dataset.rows), protocol.benchmark.latency_sample_size),
        'warmup_max': min(len(dataset.rows), protocol.benchmark.warmup_sentences),
        'latency_repeats': protocol.benchmark.latency_repeats,
    }
    for name, dataset in datasets.items()
])
workload

,test_set,quality_sentences,quality_passes,latency_sample_max,warmup_max,latency_repeats
0,general_test,1012,1,128,32,5
1,it_test,13851,1,128,32,5


## 4. Lệnh chạy baseline

Cell dưới được đặt `False` để mở notebook không tự khởi chạy nhiều giờ. Khi đã sẵn sàng chạy từ notebook, đổi biến thành `True`. Runner luôn gọi lại Phase 05 gate trước khi tải model. Trong lúc chạy, thư mục `.in_progress` có log; chỉ sau khi đủ sáu artifact, runner mới đổi tên thành thư mục kết quả chính thức.

In [ ]:
direction = 'en_to_vi'
final_dir = RUN_ROOT / direction
active_dir = RUN_ROOT / f'{direction}.in_progress'
RUN_FULL_BASELINE = False
if active_dir.is_dir():
    print('Đã có run dở dang. Đọc cell trạng thái/log; không khởi chạy chồng lên run này.')
elif final_dir.is_dir():
    print('Baseline EnViT5 EN→VI đã hoàn thành. Đọc cell kết quả phía dưới.')
elif RUN_FULL_BASELINE:
    command = [
        sys.executable, 'tools/run_envit5_baseline.py',
        '--direction', 'en_to_vi', '--device', 'cpu',
        '--progress-interval', '250', '--preview', '2',
    ]
    result = subprocess.run(
        command, cwd=ROOT, text=True, capture_output=True,
        encoding='utf-8', errors='replace',
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f'Exit code: {result.returncode}')
else:
    print('Chưa chạy. Đặt RUN_FULL_BASELINE = True để bắt đầu baseline này.')

## 5. Theo dõi trạng thái và log

Cell này chỉ đọc file. Nó hoạt động cả trước khi chạy, khi đang chạy và sau khi hoàn thành.

In [ ]:
direction = 'en_to_vi'
final_dir = RUN_ROOT / direction
active_dir = RUN_ROOT / f'{direction}.in_progress'
run_dir = final_dir if final_dir.is_dir() else active_dir

if not run_dir.is_dir():
    print('Chưa có run EnViT5 EN→VI v2.')
else:
    state = 'COMPLETED' if run_dir == final_dir else 'IN PROGRESS — chưa phải kết quả cuối'
    print(f'State: {state}')
    print('Artifacts:', sorted(path.name for path in run_dir.iterdir()))
    log = run_dir / 'run.log'
    if log.is_file():
        print('\n'.join(log.read_text(encoding='utf-8').splitlines()[-20:]))

## 6. Đọc prediction, metric và benchmark sau khi hoàn thành

Không có số liệu giả trong notebook. Nếu run chưa hoàn thành, cell sẽ báo trạng thái thay vì tự tạo chrF++, SacreBLEU, RAM hoặc latency.

In [ ]:
if not final_dir.is_dir():
    print('Metrics chưa có: chờ run EN→VI hoàn thành và được commit thành thư mục kết quả.')
else:
    metrics = json.loads((final_dir / 'metrics.json').read_text(encoding='utf-8'))['by_test_set']
    benchmark = json.loads((final_dir / 'benchmark.json').read_text(encoding='utf-8'))['by_test_set']
    predictions = [json.loads(line) for line in (final_dir / 'predictions.jsonl').read_text(encoding='utf-8').splitlines()]
    display(pd.DataFrame(predictions)[['test_set', 'row_id', 'source', 'reference', 'prediction']].head(8))
    display(pd.DataFrame(metrics).T.rename_axis('test_set'))
    display(pd.DataFrame(benchmark).T.rename_axis('test_set'))

## 7. Biểu đồ General Test và IT Test

Hai cột được giữ riêng. Chỉ khi cả hai có metric hoàn chỉnh thì biểu đồ mới xuất hiện; không tính trung bình giữa hai miền.

In [ ]:
if final_dir.is_dir():
    metric_table = pd.DataFrame(metrics).T[['chrF++', 'sacreBLEU']]
    ax = metric_table.plot.bar(rot=0, figsize=(8, 4), title='EnViT5 EN→VI: General Test và IT Test')
    ax.set_xlabel('Test set')
    ax.set_ylabel('Score')
else:
    print('Biểu đồ sẽ xuất hiện sau khi metrics.json được tạo.')

## 8. Unit test không tải checkpoint

Các test kiểm tra adapter, protocol v2 và contract artifact. Chúng không thay thế inference thật; kết quả baseline chỉ hợp lệ khi runner hoàn thành toàn bộ artifact.

In [ ]:
for pattern in ['test_core_shared.py', 'test_envit5_adapter.py', 'test_runner_contract.py']:
    result = subprocess.run(
        [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', pattern],
        cwd=ROOT, text=True, capture_output=True,
    )
    print(result.stdout + result.stderr)
    assert result.returncode == 0, pattern